# InHire Jobs Radar
Scanner automático de vagas em empresas que usam InHire.

Funcionalidades:
- Descobre empresas válidas
- Coleta vagas via API
- Filtra vagas de dados
- Detecta novas vagas
- Salva CSV e histórico

In [ ]:
import requests
import pandas as pd
import json
import unicodedata
import time
from datetime import datetime

## Normalização de texto

In [ ]:
def normalize(text):
    if not text:
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = text.encode('ascii', 'ignore').decode('utf-8')
    return text

## Palavras-chave

In [ ]:
KEYWORDS = [
 'data',
 'dados',
 'analista de dados',
 'data analyst',
 'analytics',
 'bi',
 'business intelligence',
 'cientista de dados',
 'data scientist',
 'engenheiro de dados',
 'data engineer'
]

## Empresas candidatas

In [ ]:
CANDIDATES = [
"agenciacriativa","agrosearch","alice","alun","amcom","appcargo",
"auvotecnologia","bankme","betaonline","brasilparalelo","ceisc",
"cerc","cielo","coopersystem","cora","crown","db1","deloitte",
"digix","dito","docket","evolux","fitenergia","flash","flutterbrazil",
"fretadao","gex","grupoescalar","gx2","holu","infotecbrasil",
"kanastra","kpmg","linkapi","livemode","magalu","magazord",
"milvus","mjv","nomadglobal","olist","onfly","openlabs",
"orizon","pagar-me","paketa","paytrack","pier","pontte",
"premiersoft","qive","radix","rpo-recargapay","residclub",
"samsungsds","sanar","sharepeoplehub","shareprime","supertex",
"sylvamo","sympla","talentt","talentx","tripla","unimar",
"upflux","v360","v4company","vitru","warren","xerpa","zig",
"contabilizei","kiwify","loft","nubank","creditas","ifood","stone","loggi"
]

## Verificar empresas válidas

In [ ]:
def validate_companies():
    valid = []
    for c in CANDIDATES:
        url = f"https://api.inhire.app/tenants/public/{c}"
        try:
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                print("Empresa válida:", c)
                valid.append(c)
        except:
            pass
    return valid

COMPANIES = validate_companies()
print("Total empresas válidas:", len(COMPANIES))

## Buscar vagas via API

In [ ]:
def fetch_jobs(company):
    jobs = []
    url = "https://api.inhire.app/job-posts/public"
    headers = {
        "X-Tenant": company,
        "User-Agent": "Mozilla/5.0"
    }

    page = 0

    while True:
        params = {"page": page, "size": 50}

        r = requests.get(url, headers=headers, params=params)

        if r.status_code != 200:
            break

        data = r.json()

        if 'content' not in data:
            break

        content = data['content']

        if len(content) == 0:
            break

        for job in content:
            job_id = job.get('id','')
            title = job.get('title','')
            slug = job.get('slug','')

            link = f"https://{company}.inhire.app/vagas/{job_id}/{slug}"

            jobs.append({
                'empresa': company,
                'titulo': title,
                'link': link
            })

        page += 1

    return jobs

## Scanner principal

In [ ]:
all_jobs = []

for company in COMPANIES:
    print("Escaneando:", company)
    jobs = fetch_jobs(company)
    all_jobs.extend(jobs)
    time.sleep(0.3)

print("Total vagas coletadas:", len(all_jobs))

## Filtrar vagas de dados

In [ ]:
def is_data_job(title):
    t = normalize(title)
    for k in KEYWORDS:
        if normalize(k) in t:
            return True
    return False

data_jobs = [j for j in all_jobs if is_data_job(j['titulo'])]

print("Vagas de dados encontradas:", len(data_jobs))

## Detectar vagas novas

In [ ]:
try:
    with open('jobs_history.json') as f:
        history = json.load(f)
except:
    history = []

old_links = set([j['link'] for j in history])

new_jobs = [j for j in data_jobs if j['link'] not in old_links]

print("Novas vagas detectadas:", len(new_jobs))

for j in new_jobs:
    print(j['empresa'], '-', j['titulo'])

## Salvar resultados

In [ ]:
df = pd.DataFrame(data_jobs)

df.to_csv('vagas_data_analyst.csv', index=False)

with open('jobs_history.json','w') as f:
    json.dump(data_jobs, f, indent=2)

print('CSV salvo: vagas_data_analyst.csv')